In [1]:
import os
import json
import time
import argparse
from typing import Tuple, Optional

import numpy as np
from scipy.optimize import minimize

#few and SEF imports
from few.waveform import GenerateEMRIWaveform
from few.waveform.waveform import SuperKludgeWaveform

from few.trajectory.inspiral import EMRIInspiral
from few.trajectory.ode.flux import KerrEccEqFlux
from few.utils.geodesic import get_fundamental_frequencies
from scipy.interpolate import CubicSpline
from scipy.integrate import cumulative_trapezoid
from few.utils.constants import MTSUN_SI

from fastlisaresponse import ResponseWrapper
from lisatools.detector import EqualArmlengthOrbits
from lisatools.sensitivity import get_sensitivity, A1TDISens, E1TDISens, T1TDISens
from stableemrifisher.utils import generate_PSD, inner_product
from stableemrifisher.fisher import StableEMRIFisher


from config_paris import Config, ObjectiveTracker
from misc import _is_pos_def, check_and_clip_prior, _clip_physical_params_intrinsic,compute_fft_with_windowing, calculate_optimal_snr_0pa_vs_1pa, compute_fisher_parallelotope,covariance_from_fisher_parallelotope


startup


In [6]:
# -----------------------------
# PARIS global context (picklable functions require module scope)
# -----------------------------..l
_PARIS_REF_CENTER = None          # type: Optional[np.ndarray]
_PARIS_SPREAD_SCALE = None        # type: Optional[float]
_PARIS_OBJECTIVE = None           # type: Optional[callable]
_PARIS_TARGET_KIND = None         # type: Optional[str]  # 'optimal_snr', 'optimal_snr_phase_max', 'phase_match'
_PARIS_EARLY_STOP_HIT = False

# Fisher-parallelotope affine prior (primary for this script)
_PARIS_AFFINE_CENTER = None       # type: Optional[np.ndarray]
_PARIS_AFFINE_Q = None            # type: Optional[np.ndarray]
_PARIS_AFFINE_B = None            # type: Optional[np.ndarray]
_PARIS_DIM = None                 # type: Optional[int]
_PARIS_USE_ELLIPSE = True

# Target SNR for Fisher scaling
_TARGET_SNR = Config()._TARGET_SNR

params_to_infer = Config().param_names_to_infer
dt = Config().dt
T = Config().T
chi2 = Config().chi2
dev_1 = Config().dev_1
dev_2 = Config().dev_2

params_to_infer

['m1', 'm2', 'a', 'p0', 'e0', 'qS', 'phiS', 'Phi_phi0', 'Phi_r0']

In [4]:
def paris_prior_transform(u):
    """Prior transform using Fisher-parallelotope when configured.

    If affine parameters are set: theta = center + Q @ (b * t), with t = 2u-1.
    Otherwise falls back to multiplicative band around _PARIS_REF_CENTER.
    """
    u = np.asarray(u, dtype=float)
    if _PARIS_AFFINE_CENTER is not None and _PARIS_AFFINE_Q is not None and _PARIS_AFFINE_B is not None:
        center = _PARIS_AFFINE_CENTER
        Q = _PARIS_AFFINE_Q
        b = _PARIS_AFFINE_B
        dim = Q.shape[0]

        def map_one(u1):
            t = 2.0 * np.asarray(u1)[:dim] - 1.0
            return center + Q @ (b * t)

        if u.ndim == 1:
            theta = map_one(u)
            # if theta.shape[0] == 5:
            return _clip_physical_params_intrinsic(theta)
            # else:
            #     return check_and_clip_prior(theta, params_to_infer)
        else:
            out = np.zeros((u.shape[0], dim), dtype=float)
            for i in range(u.shape[0]):
                out[i] = map_one(u[i])
            return _clip_physical_params_intrinsic(out)
            # else:
            #     return check_and_clip_prior(out, params_to_infer)

    # Legacy multiplicative band (fallback)
    ref = _PARIS_REF_CENTER
    s = _PARIS_SPREAD_SCALE
    u = np.asarray(u)
    return ref * (1 - s + u * 2 * s)


def paris_inverse_prior_transform(params):
    """Inverse of paris_prior_transform.

    For affine Fisher mapping: t = diag(1/b) Q^T (theta - center), u = 0.5*(t+1).
    Falls back to multiplicative inverse if affine not configured.
    """
    theta = np.asarray(params, dtype=float)
    if _PARIS_AFFINE_CENTER is not None and _PARIS_AFFINE_Q is not None and _PARIS_AFFINE_B is not None:
        center = _PARIS_AFFINE_CENTER
        Q = _PARIS_AFFINE_Q
        b = _PARIS_AFFINE_B
        inv_b = 1.0 / b

        def inv_one(th):
            d = np.asarray(th) - center
            t = (Q.T @ d) * inv_b
            return 0.5 * (t + 1.0)

        if theta.ndim == 1:
            return inv_one(theta)
        out = np.zeros_like(theta, dtype=float)
        for i in range(theta.shape[0]):
            out[i] = inv_one(theta[i])
        return out

    ref = _PARIS_REF_CENTER
    s = _PARIS_SPREAD_SCALE
    return (theta / ref - (1 - s)) / (2 * s)


def paris_log_density(params):
    """Top-level log-density (actually score) wrapper with early-stop.

    - Receives physical parameters (after prior_transform) per parismc contract.
    - Returns a scalar/array of scores (larger is better). After early-stop trigger,
      subsequent evaluations return -inf to end sampling quickly.
    """
    global _PARIS_EARLY_STOP_HIT

    params = np.asarray(params)

    def eval_one(x):
        global _PARIS_EARLY_STOP_HIT
        if _PARIS_EARLY_STOP_HIT:
            return float('-inf')
        try:    
            val = float(_PARIS_OBJECTIVE(x))
        except Exception:
            return float('-inf')
        # Early-stop policy per user spec
        if _PARIS_TARGET_KIND in ('optimal_snr', 'optimal_snr_phase_max'):
            if val >= 19.0:
                _PARIS_EARLY_STOP_HIT = True
                try:
                    print(f"[EARLY-STOP] SNR {val:.6f} >= 19; future calls => -inf")
                except Exception:
                    pass
        elif _PARIS_TARGET_KIND == 'phase_match':
            # score = -phase_diff; trigger when phase_diff < 1.5 => score > -1.5
            if val > -1.5:
                _PARIS_EARLY_STOP_HIT = True
                try:
                    print(f"[EARLY-STOP] phase-diff {-val:.6f} < 1.5; future calls => -inf")
                except Exception:
                    pass
        return val

    if params.ndim == 1:
        return eval_one(params)
    out = np.zeros(params.shape[0], dtype=float)
    for i in range(params.shape[0]):
        out[i] = eval_one(params[i])
    return out


In [7]:
def build_waveform_response(T: float, dt: float, use_gpu: bool = False) -> ResponseWrapper:
    """Create a LISA ResponseWrapper consistent with existing modules."""

    sum_kwargs = dict(pad_output=False, odd_len=True)
    waveform_model = GenerateEMRIWaveform(SuperKludgeWaveform, sum_kwargs=sum_kwargs, return_list=False)

    t0 = 10000.0
    tdi_gen = "2nd generation"
    order = 20
    index_lambda = 8  # phiS
    index_beta = 7    # qS

    response = ResponseWrapper(
        waveform_gen=waveform_model,
        Tobs=T,
        t0=t0,
        dt=dt,
        index_lambda=index_lambda,
        index_beta=index_beta,
        flip_hx=True,
      #  use_gpu=use_gpu,
        is_ecliptic_latitude=False,
        remove_garbage="zero",
        orbits=EqualArmlengthOrbits(use_gpu=use_gpu),
        order=order,
        tdi=tdi_gen,
        tdi_chan="AET",
    )
    print("[INFO] Finished loading modules and building ResponseWrapper")
    return response

def prepare_true_waveform(signal_row: np.ndarray, add_kwargs: dict, use_gpu: bool = False):
    """
    Build fiducial 1PA waveform, PSD, and FFT from a signal parameter row.

    signal_row columns:
      [m1, m2, a, p0, e0, Y0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0]
    """
    (
        m1, m2, a, p0, e0, Y0,
        dist, qS, phiS, qK, phiK,
        Phi_phi0, Phi_theta0, Phi_r0
    ) = signal_row

    waveform_response = build_waveform_response(T=T, dt=dt, use_gpu=use_gpu)

    chi2 = add_kwargs.get('chi2')
    deviation_included = add_kwargs.get('deviation_included', True)
    evolve_1PA = add_kwargs.get('evolve_1PA',)
    evolve_primary = add_kwargs.get('evolve_primary', False)
    evolve_2PA = add_kwargs.get('evolve_2PA',False)
    dev_1 = add_kwargs.get('dev_a')
    dev_2 = add_kwargs.get('dev_b')

    wave_params = [
        m1, m2, a, p0, e0, Y0,
        dist, qS, phiS, qK, phiK,
        Phi_phi0, Phi_theta0, Phi_r0,chi2, evolve_1PA, evolve_primary, evolve_2PA, deviation_included, dev_1, dev_2
    ]
    
    emri_kwargs = {"T": T, "dt": dt, '1PA': evolve_1PA,'chi2': chi2, 'evolve_primary': evolve_primary,
                    '2PA': evolve_2PA,'deviation_included': deviation_included,'dev_a': dev_1, 'dev_b': dev_2}

    waveform_true = waveform_response(*wave_params, **emri_kwargs)

    channels = [A1TDISens, E1TDISens, T1TDISens]
    noise_kwargs = [{"sens_fn": ch} for ch in channels]
    PSD_funcs = generate_PSD(
        waveform=waveform_true,
        dt=dt,
        noise_PSD=get_sensitivity,
        channels=channels,
        noise_kwargs=noise_kwargs,
        use_gpu=use_gpu,
    )

    # Verify SNR level (grid builder normalized dist to target PA2 SNR already)
    snr = float(np.sqrt(inner_product(waveform_true, waveform_true, PSD_funcs, dt, use_gpu=use_gpu)))
    print(f"[TRUE] SNR: {snr:.6f}")

    waveform_true_fft = compute_fft_with_windowing(waveform_true, dt, use_gpu=use_gpu)
    N_fiducial = len(waveform_true[0])
    print("[INFO] Finished preparing true waveform (GPU)")

    return {
        'm1': m1, 'm2': m2, 'a': a, 'p0': p0, 'e0': e0, 'Y0': Y0,
        'dist': dist, 'qS': qS, 'phiS': phiS, 'qK': qK, 'phiK': phiK,
        'Phi_phi0': Phi_phi0, 'Phi_theta0': Phi_theta0, 'Phi_r0': Phi_r0,
        'dt': dt, 'T': T, 'chi2': chi2,
        'dev_1': dev_1, 'dev_2': dev_2,
        'waveform_response': waveform_response,
        'PSD_funcs': PSD_funcs,
        'waveform_true_fft': waveform_true_fft,
        'N_fiducial': N_fiducial,
        'snr': snr,
    }


In [ ]:
# this need a lot of work to be implemented
def objective_factory(target_func: str,
                      ctx: dict,
                      phase_max: bool = False,
                      use_gpu_for_snr: bool = True,
                      infer_deviation_included: bool = False,
                      only_intrinsic_params: bool = False,
                      add_args: dict = None) -> callable:
    """
    Build a score(theta) where larger is better for all targets.
    - 'optimal_snr' and 'optimal_snr_phase_max': score = optimal SNR (maximize)
    - 'phase_match': score = -phase_diff_metric (maximize score => minimize phase diff)
    """
    # Only needed for SNR-based objective
    if target_func in ('optimal_snr', 'optimal_snr_phase_max'):
        fixed = {
            'waveform_response': ctx['waveform_response'],
            'PSD': ctx['PSD_funcs'],
            'dt': ctx['dt'],
            'T': ctx['T'],
            'N_fiducial': ctx['N_fiducial'],
            'waveform_true_fft': ctx['waveform_true_fft'],
            'xp': np,
            'use_gpu': bool(use_gpu_for_snr),
        }

    def score_optimal_snr(theta: np.ndarray) -> float:
   
   
        if only_intrinsic_params == True:
            m1, m2, a, p0, e0 = theta[:5]
            val = calculate_optimal_snr_0pa_vs_1pa(
                m1, m2, a, p0, e0, ctx['Y0'],ctx['dist'],ctx['qS'],ctx['phiS'], ctx['qK'], ctx['phiK'], 
                ctx['Phi_phi0'], ctx['Phi_theta0'], ctx['Phi_r0'],add_args,
                maximize_phase=bool(phase_max),
                **fixed,
            )
        else:
            if infer_deviation_included:
                m1, m2, a, p0, e0,qS,phiS,Phi_phi0,Phi_r0,dev_1,dev_2 = theta[:6]
                add_args['dev_1'] = dev_1
                add_args['dev_2'] = dev_2
                val = calculate_optimal_snr_0pa_vs_1pa(
                    m1, m2, a, p0, e0, ctx['Y0'],ctx['dist'],qS,phiS, ctx['qK'], ctx['phiK'], 
                    Phi_phi0, ctx['Phi_theta0'], Phi_r0,add_args,
                    maximize_phase=bool(phase_max),
                    **fixed)
                print(val, 'param', repr(theta))
        # Score is the SNR itself (maximize)
        return float(val)

    # Phase-match uses trajectory frequency metric from phase-match.py logic
    # Here we re-implement a light-weight metric using FEW's fundamental frequencies.
  

    # Precompute 1PA Omega_phi(t) interpolation and frequency weights
    # Build 1PA trajectory from the signal row
    SK_traj_1PA = EMRIInspiral(func=KerrEccEqFlux)  # For 0PA vs 1PA metric we still compare to the same t-grid
    # Note: We only need 1PA ref already embedded in ctx? We rebuild Omega_phi_1PA curve here robustly
    # However ctx doesn’t include (t,p,e,x) of 1PA; reconstruct for phase metric
    from few.trajectory.inspiral import EMRIInspiral as EMRIInspiralFull
    from few.trajectory.ode.flux import SuperKludgeFlux

    SK_traj_true = EMRIInspiralFull(func=SuperKludgeFlux)

    # do we need to implement it inside the function

    traj_ref = SK_traj_true.get_inspiral(
        ctx['m1'], ctx['m2'], ctx['a'], ctx['p0'], ctx['e0'], ctx['Y0'],
        ctx['chi2'], True, False, False,False,False,0,0,Phi_phi0 = ctx['Phi_phi0'], Phi_theta0 = ctx['Phi_theta0'], Phi_r0 = ctx['Phi_r0'],
        T=ctx['T'], dt=ctx['dt'], err=1e-11, DENSE_STEPPING=False,
        buffer_length=1000, integrate_backwards=False,
        max_step_size=None,
    )
    t_ref, p_ref, e_ref, x_ref = traj_ref[0], traj_ref[1], traj_ref[2], traj_ref[3]
    Omega_phi_ref, _, _ = get_fundamental_frequencies(ctx['a'], p_ref, e_ref, x_ref)
    # Common time grid and 1PA omega interpolation
    t_common = np.linspace(t_ref.min(), t_ref.max(), 1000)
    Omega_phi_1PA_interp = CubicSpline(t_ref, Omega_phi_ref)(t_common)

    # Frequency weighting w(t) ~ sum 1/S_n(f_gw)
    m_mode = 2
    Msec = (ctx['m1'] + ctx['m2']) * MTSUN_SI
    Omega2_SI = Omega_phi_1PA_interp / Msec
    f_gw = m_mode * Omega2_SI / (2.0 * np.pi)
    w = np.zeros_like(f_gw)
    for ch in (A1TDISens, E1TDISens, T1TDISens):
        Sn = get_sensitivity(f_gw, sens_fn=ch)
        Sn = np.maximum(Sn, 1e-60)
        w += 1.0 / Sn

    def phase_metric_for_theta(theta: np.ndarray) -> float:
        # Supports both 0PA against 1PA reference
        if only_intrinsic_params == True:
            m1, m2, a, p0, e0 = theta[:5]
            add_args['evolve_1PA'] = False
            use_1pa = False
        else:
            if infer_deviation_included:
                m1, m2, a, p0, e0,qS,phiS,Phi_phi0,Phi_r0,dev_1,dev_2 = theta[:6]
                add_args['dev_1'] = dev_1
                add_args['dev_2'] = dev_2
                add_args['deviation_included'] = True
                add_args['evolve_1PA'] = False
                use_1pa = False
            else:
                m1, m2, a, p0, e0,qS,phiS,Phi_phi0,Phi_r0 = theta[:6]
                add_args['dev_1'] = dev_1
                add_args['dev_2'] = dev_2
                add_args['deviation_included'] = True
                add_args['evolve_1PA'] = False
                use_1pa = False

        # Build trajectory for the chosen template order (0PA or 1PA) without 2PA
        SK_traj_tmpl = EMRIInspiral(func=KerrEccEqFlux)

        traj_0 = SK_traj_tmpl.get_inspiral(
            m1, m2, a, p0, e0, ctx['Y0'], add_args['chi2'], use_1pa, False,  False, False,False,False,add_args['dev_1'],add_args['dev_1'],
            Phi_phi0 = Phi_phi0, Phi_theta0 = ctx['Phi_theta0'], Phi_r0 = Phi_r0,
        T=ctx['T'], dt=ctx['dt'], err=1e-11, DENSE_STEPPING=False,
        buffer_length=1000, integrate_backwards=False,
        max_step_size=None,)

        t0, p0_arr, e0_arr, x0 = traj_0[0], traj_0[1], traj_0[2], traj_0[3]
        Omega_phi_0, _, _ = get_fundamental_frequencies(a, p0_arr, e0_arr, x0)
        # Interpolate to common grid
        Omega_phi_0_interp = CubicSpline(t0, Omega_phi_0)(t_common)
        # Weighted phase difference metric
        dOmega_geo = Omega_phi_0_interp - Omega_phi_1PA_interp
        dOmega_SI = dOmega_geo / Msec
        dphi = np.concatenate([[0.0], m_mode * cumulative_trapezoid(dOmega_SI, t_common)])
        num = np.trapz(w * dphi**2, t_common)
        den = np.trapz(w, t_common)
        res = np.sqrt(num / den)
        # Print theta as a copyable NumPy array with high precision
        theta_repr = "np.array(" + np.array2string(
            theta,
            formatter={'float_kind': lambda x: f"{x:.12g}"},
            separator=', '
        ) + ")"
        print(f"{res:.6g}", 'param', theta_repr)
        return res

    def score_phase_match(theta: np.ndarray) -> float:
        # Lower phase difference should be better => use negative for a larger-is-better score
        return -float(phase_metric_for_theta(theta))

    if target_func in ('optimal_snr', 'optimal_snr_phase_max'):
        return score_optimal_snr
    elif target_func == 'phase_match':
        return score_phase_match
    else:
        raise ValueError(f"Unknown target_func: {target_func}")
    